# Churn baseline models

## 01. Executive overview

A retention team would use an episode-level churn score to decide which active subscriptions
warrant review before a qualifying termination. One customer can hold several distinct
contractual episodes. Alerts here count **episode observations**, so they cannot be read as
unique customers or staffing requirements.

The dummy supplies a no-ranking reference tied to training prevalence. L2 logistic regression
tests whether a simple weighted combination of the approved features can improve that ranking.
In the retained Fieldwise Cloud experiment neither logistic configuration beat prevalence on
average precision. The unweighted fit flagged 510 of 717 observations to find all eight
positives, generating 502 false alerts. Validation was synthetic, and the separate reserve
remains unscored. These results describe the reference experiment’s limits for prioritisation.

The notebook renders checksummed artefacts from the
[Phase 06 protocol](../../docs/05_ml/phase06_churn_evaluation_protocol.md).
See the [notebook guide](../README.md#phase-06-churn-modelling) for local rendering.

In [ ]:
import os
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import PercentFormatter

from nowa_customer_intelligence.models.churn.research import DEFAULT_WORKSPACE, load_reports

repository = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
results_dir = Path(os.environ.get("NCI_CHURN_RESULTS", repository / DEFAULT_WORKSPACE / "results"))
baseline, advanced = load_reports(results_dir)
design = baseline["design"]
plt.style.use("seaborn-v0_8-whitegrid")
palette = ["#65758b", "#245293", "#138577", "#854ba5", "#ae5c27", "#a23c55", "#151e37"]
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold", "font.size": 10})


def metric_table(candidates):
    rows = []
    for item in candidates:
        row = item["validation"]
        rows.append(
            {
                "Model": item["name"],
                "Threshold": item["threshold"],
                "AP": row["average_precision"],
                "ROC-AUC": row["roc_auc"],
                "Precision": row["precision"],
                "Recall": row["recall"],
                "F1": row["f1"],
                "Brier": row["brier"],
                "Positives": row["positives"],
                "TN": row["tn"],
                "FP": row["fp"],
                "FN": row["fn"],
                "TP": row["tp"],
            }
        )
    return pd.DataFrame(rows).set_index("Model")


def show():
    plt.tight_layout()
    plt.show()
    plt.close("all")


candidates = baseline["candidates"]
logistic = next(item for item in candidates if item["name"] == baseline["selected"])
counts = pd.DataFrame(design["support"]).T


def operational_table(candidates):
    rows = []
    for candidate in candidates:
        m = candidate["validation"]
        alerts = m["tp"] + m["fp"]
        assert m["tn"] + alerts + m["fn"] == m["rows"]
        assert m["tp"] + m["fn"] == m["positives"]
        assert np.isclose(m["precision"], m["tp"] / alerts if alerts else 0)
        assert np.isclose(m["recall"], m["tp"] / m["positives"] if m["positives"] else 0)
        rows.append(
            {
                "Model": candidate["name"],
                "Rows": m["rows"],
                "Alerts": alerts,
                "Alert rate": alerts / m["rows"],
                "True alerts": m["tp"],
                "False alerts": m["fp"],
                "Missed churn": m["fn"],
                "False discovery": m["fp"] / alerts if alerts else np.nan,
                "Precision": m["precision"],
                "Recall": m["recall"],
            }
        )
    return pd.DataFrame(rows).set_index("Model")


def plot_alerts(table):
    fig, ax = plt.subplots(figsize=(10, 1.8 + 0.52 * len(table)))
    columns = ["False alerts", "True alerts"]
    table[columns].plot.barh(stacked=True, ax=ax, color=["#a74932", "#167568"], width=0.58)
    for index, (alerts, rate) in enumerate(zip(table["Alerts"], table["Alert rate"], strict=True)):
        ax.text(
            alerts + table["Rows"].max() * 0.016, index, f"{alerts:,} ({rate:.1%})", va="center"
        )
    ax.set(
        title="Validation review queue at each frozen threshold",
        xlabel="Alerted episode observations",
        ylabel="",
        xlim=(0, table["Rows"].max() * 1.22),
    )
    ax.invert_yaxis()
    ax.grid(False, axis="y")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False)
    show()


def cohort_table(records):
    table = pd.DataFrame(records).rename(
        columns={
            "cohort": "Cohort",
            "level": "Group",
            "rows": "Rows",
            "positives": "Positive rows",
            "positive_accounts": "Positive accounts",
            "tp": "True alerts",
            "fp": "False alerts",
            "fn": "Missed churn",
        }
    )
    table["Alert rate"] = (table["True alerts"] + table["False alerts"]) / table["Rows"]
    table["False alerts / all rows"] = table["False alerts"] / table["Rows"]
    return table.drop(columns="recall")


def plot_confusion(candidates):
    fig, axes = plt.subplots(1, len(candidates), figsize=(4 * len(candidates), 3.5), squeeze=False)
    for ax, candidate in zip(axes[0], candidates, strict=True):
        m = candidate["validation"]
        matrix = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
        # Shade within observed classes so rare positive cases remain visible.
        fractions = matrix / matrix.sum(axis=1, keepdims=True)
        ax.imshow(fractions, cmap="Blues", vmin=0, vmax=1)
        ax.grid(False)
        for (y, x), value in np.ndenumerate(matrix):
            fraction = fractions[y, x]
            ax.text(
                x,
                y,
                f"{value:,}\n{fraction:.1%}",
                ha="center",
                va="center",
                color="white" if fraction > 0.56 else "#131f31",
            )
        ax.set(
            xticks=[0, 1],
            xticklabels=["No alert", "Alert"],
            yticks=[0, 1],
            yticklabels=["No churn", "Churn"],
            xlabel="Frozen decision",
            title=candidate["name"],
            ylabel="Resolved outcome",
        )
    fig.suptitle("Validation confusion matrices · counts and within-class percentages")
    show()


operations = operational_table(candidates)
metrics = metric_table(candidates).join(operations[["Alerts", "Alert rate"]])
chosen_metrics = logistic["validation"]


def format_table(frame, formats, na_rep="—"):
    """Format display values while retaining numeric source columns."""
    display_frame = frame.copy()
    for column, pattern in formats.items():
        display_frame[column] = frame[column].map(pattern.format, na_action="ignore").fillna(na_rep)
    return display_frame

## 02. Prediction target

The unit is an eligible canonical subscription episode observed at a UTC snapshot. A later
reactivation is another episode of the same customer, with its own identity. Each prediction uses
facts known at the observation instant. Identifiers, observation timestamps and ascertainment are
separate from the feature matrix.

A confirmed qualified effective termination after the observation and by the end of the 30-day
horizon is positive. A confirmed negative requires evidence that the horizon passed without that
event. Right-censored outcomes lack complete follow-up. Pending and ineligible outcomes also stay
outside supervised fitting. The observation-time rules require an established, active episode with
usable evidence and enough history. The 90-day history and 30-day horizon are fictional experiment
parameters.

## 03. Modelling dataset

Which observations reach modelling? Follow the seed-42 population from candidate snapshots
through as-known eligibility to label resolution. The left panel uses the same candidate base
at each step. The right panel splits only resolved rows into confirmed outcomes. The two
panels have different count axes. Censoring leaves an otherwise eligible row without a complete
label at extraction. Insufficient history excludes it from the earlier prediction roster.

In [ ]:
inventory = design["reference_inventory"]
total, eligible, resolved = (
    inventory[k] for k in ("candidates", "prediction_eligible", "resolved_eligible")
)
positive = inventory["outcomes"]["confirmed_positive"]
negative = inventory["outcomes"].get("confirmed_negative", resolved - positive)
censored = inventory["outcomes"].get("right_censored", 0)
history = inventory["prediction_exclusions"].get("insufficient_observation_history", 0)
assert positive + negative == resolved and 0 <= resolved <= eligible <= total
population = pd.DataFrame(
    [
        ("Candidate observations", total, total),
        ("Prediction eligible", eligible, total),
        ("Excluded for insufficient history", history, total),
        ("Resolved binary outcome", resolved, eligible),
        ("Right-censored eligible rows", censored, eligible),
        ("Confirmed negative", negative, resolved),
        ("Confirmed positive", positive, resolved),
    ],
    columns=["Stage or outcome", "Rows", "Denominator"],
)
population["Share of denominator"] = population["Rows"] / population["Denominator"]
display(
    format_table(
        population, {"Rows": "{:,}", "Denominator": "{:,}", "Share of denominator": "{:.2%}"}
    )
)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.1))
axes[0].barh(
    ["Candidate", "Eligible", "Resolved"],
    [total, eligible, resolved],
    color=["#6d7e92", "#4777a3", "#255889"],
)
axes[1].barh(["Negative", "Positive"], [negative, positive], color=["#6d7e92", "#a74932"])
for axis, title in zip(
    axes, ("Seed-42 eligibility and resolution", "Resolved class counts"), strict=True
):
    axis.grid(False, axis="y")
    axis.bar_label(axis.containers[0], fmt=lambda value: f"{value:,.0f}", padding=4)
    axis.set(
        title=title,
        xlabel="Episode observations",
        xlim=(0, max(axis.containers[0].datavalues) * 1.21),
    )
    axis.invert_yaxis()
show()

These counts refer to snapshots, so one episode can contribute at several times. The
resolved positives are rare even before the group and maturity rules reduce the validation
set. A blocked eligibility or missing label has a different meaning from a confirmed negative.

## 04. Temporal validation strategy

How does the design approximate a later scoring period? The hash assigns each seed-42 customer
to one development side. All episodes of the same account, including reactivations, stay
together. A row contributes only inside its account's assigned date window. Its whole outcome
horizon and ascertainment must precede the exclusive right boundary. This removes labels
that would cross into the next window. Different accounts can share calendar periods in
their lookback, and rows within an account can share evidence.

The shaded bands show permitted observation windows. Thick bars show retained observation
spans. Vertical lines are the exclusive maturity boundaries. September 2023 onward from
seed 42 was examined in Batch 09 and is development evidence only. The later seed-2027
band represents a separately generated reserve, which remains unscored.

In [ ]:
limits = pd.to_datetime(pd.Series(design["boundaries"]))
start = pd.Timestamp("2021-04-01", tz="UTC")
windows = [
    (start, limits["training_end"]),
    (limits["training_end"], limits["validation_end"]),
    (limits["validation_end"], limits["holdout_end"]),
]
fig, ax = plt.subplots(figsize=(12, 3.4))
for index, (name, window) in enumerate(zip(counts.index, windows, strict=True)):
    left, right = window
    ax.barh(index, right - left, left=left, height=0.6, color=palette[index], alpha=0.2)
    if counts.at[name, "rows"]:
        first, last = (
            pd.Timestamp(counts.at[name, field])
            for field in ("first_observation", "last_observation")
        )
        ax.plot(
            [first, last],
            [index, index],
            linewidth=5,
            color=palette[index],
            marker="|",
            label=f"{name.title()} retained observation span",
        )
for boundary in limits:
    ax.axvline(boundary, ls="--", lw=1, color=palette[0])
ax.set(
    yticks=range(3),
    yticklabels=["Training", "Validation", "Reserved"],
    xlabel="UTC calendar",
    title="Forward evaluation windows and exclusive maturity boundaries",
)
ax.invert_yaxis()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(loc="upper right", frameon=True, facecolor="white", fontsize=8)
show()
display(
    pd.Series(
        design["development_exclusions"], name="Resolved development rows excluded"
    ).to_frame()
)

## 05. Class imbalance

How many observations and independent events support the comparison? The denominator for
prevalence is resolved rows within each partition. A positive row and a distinct positive
account answer different questions. Several snapshots could refer to one customer or event.
The support floor applies to accounts. The holdout bar uses labels for feasibility only.

In [ ]:
support_table = counts[
    [
        "rows",
        "positives",
        "negatives",
        "customers",
        "episodes",
        "positive_customers",
        "positive_episodes",
        "max_positives_per_customer",
        "prevalence",
    ]
].copy()
support_table["prevalence"] = (
    support_table["prevalence"].astype(float).map("{:.2%}".format, na_action="ignore")
)
display(support_table)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(counts.index, counts["prevalence"].astype(float), color=palette[:3])
axes[0].yaxis.set_major_formatter(PercentFormatter(1))
axes[0].set(title="Churn prevalence by partition", ylabel="Resolved observations (%)")
x = np.arange(len(counts))
axes[1].bar(
    x - 0.17,
    counts["positive_customers"].astype(int),
    width=0.34,
    label="Observed",
    color=palette[1],
)
axes[1].bar(
    x + 0.17,
    list(design["confirmation_support_minimum"].values()),
    width=0.34,
    label="Confirmation floor",
    color=palette[0],
)
axes[1].set(
    xticks=x,
    xticklabels=counts.index,
    title="Independent positive support",
    ylabel="Positive customer accounts",
)
axes[1].legend()
show()

With eight positives in 717 validation observations, predicting no churn would be **98.88%
accurate** while missing every observed event. One event changes validation recall by
12.5 percentage points. The eight positive accounts also fell short of the ten-account
confirmation floor. Apparent differences between models can depend heavily on a few ranks
or threshold decisions.

## 06. Features and preprocessing

Could the pipeline use information unavailable when a subscription is scored? The Phase 05
registry defines the ordered feature set. Missingness is measured over the prediction-eligible
roster, including eligible rows later censored. The summary groups registered variables
without repeating the [Phase 05 dictionary](../02_feature_engineering/04_feature_engineering_analysis.ipynb).

In [ ]:
feature_table = pd.DataFrame(design["features"])
feature_table["Missing (%)"] = 100 * feature_table["missing"] / eligible
features_by_group = feature_table.groupby(["group", "type"], sort=False).agg(
    Predictors=("feature", "count"),
    Nullable=("nullable", "sum"),
    Any_missing=("missing", lambda values: (values > 0).sum()),
)
categorical_count = feature_table["type"].eq("str").sum()
display(
    Markdown(
        f"**{len(feature_table)} registered predictors** · "
        f"{len(feature_table) - categorical_count} numerical and {categorical_count} categorical."
    )
)
display(features_by_group)
fig, axes = plt.subplots(1, 2, figsize=(11.4, 3.6))
missing = feature_table[feature_table["missing"] > 0].sort_values("Missing (%)")
axes[0].barh(missing["feature"].str.replace("_", " "), missing["Missing (%)"], color=palette[1])
axes[0].set(title="Incomplete features", xlabel="Prediction-eligible observations (%)")
coverage = pd.Series(design["reference_coverage"]).sort_index()
axes[1].bar(coverage.index, coverage.values, color=palette[2])
axes[1].set(
    title="Available whole months / expected months",
    xlabel="Usage coverage",
    ylabel="Prediction-eligible observations",
)
show()
display(
    pd.DataFrame(
        [
            ("Numeric", "Median imputation; scaling for logistic regression"),
            ("Nullable numeric", "Missingness flags; recorded zero activity remains zero"),
            ("Categorical", "One-hot encoding; unseen categories handled without refitting"),
            ("Constant columns", "Removed using variation measured on training only"),
            (
                "Metadata and outcomes",
                "Identifiers, timestamps and adjudication kept outside predictors",
            ),
        ],
        columns=["Input", "Pipeline handling"],
    )
)

The simulated reporting lag and short window leave many usage trends undefined. Trend features
require consecutive available months. Billing values can be absent when no qualifying invoice
exists. Exact GBP is retained until numeric model input. Learned medians, encodings, scales
and filters are fitted exclusively on training rows.

## 07. Dummy classifier

The dummy emits the same training prevalence score for all rows. It anchors validation AP at
validation prevalence and has no ranking power. At its frozen F1 threshold it flags all rows.

## 08. Logistic regression configurations

L2 shrinkage limits extreme coefficients when features are correlated or evidence is sparse.
Both configurations use the same penalty strength. Balanced weighting gives a larger
contribution to rare positive training cases and changes score scale. The class weights do
not supply additional independent events.

In [ ]:
display(
    format_table(
        pd.DataFrame(
            [
                {
                    "Configuration": c["name"],
                    "Estimator": "Constant prior"
                    if "strategy" in c["parameters"]
                    else "Logistic regression",
                    "Class weight": c["parameters"].get("class_weight") or "None",
                    "Regularisation C": c["parameters"].get("C"),
                    "Frozen threshold": c["threshold"],
                }
                for c in candidates
            ]
        ),
        {"Frozen threshold": "{:.6f}"},
        na_rep="—",
    )
)

## 09. Baseline performance dashboard

AP and ROC-AUC summarise ranking across thresholds. Brier score measures mean squared score
error. Precision, recall, F1, error counts and alert rate describe decisions at the saved
thresholds. Validation prevalence is the AP reference for an uninformative ranking.

In [ ]:
display(
    format_table(
        metrics,
        {
            "Threshold": "{:.6f}",
            "AP": "{:.5f}",
            "ROC-AUC": "{:.4f}",
            "Brier": "{:.5f}",
            "Precision": "{:.2%}",
            "Recall": "{:.2%}",
            "F1": "{:.4f}",
            "Alert rate": "{:.2%}",
        },
    )
)

display(Markdown(f"**Validation prevalence reference** {chosen_metrics['prevalence']:.3%}."))

Logistic regression reduces the number of false alerts at these thresholds, while both fits
have lower AP and ROC-AUC than the prior reference. A threshold-specific reduction in alerts
can coexist with weak overall ranking.

## 10. Precision–recall analysis

Precision is the share of alerts that correspond to churn. Recall is the share of observed
churn that is alerted. AP summarises precision over increments in recall. The dashed line
marks validation prevalence. Both panels show the same curves. The right panel zooms to
the first eight percentage points of precision so small differences near prevalence are
readable. The left panel includes the entire range and the conventional zero-recall endpoint.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6), sharex=True)
for ax, title, ceiling in zip(
    axes, ("Full precision range", "Zoom · precision 0–8%"), (1, 0.08), strict=True
):
    for item, colour, style in zip(candidates, palette[:3], ("-", "--", "-."), strict=True):
        curve = item["pr"]
        ax.step(
            curve["recall"],
            curve["precision"],
            where="post",
            label=item["name"],
            color=colour,
            ls=style,
        )
    ax.axhline(chosen_metrics["prevalence"], ls=":", color="black", label="Prevalence")
    ax.set(title=title, xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, ceiling))
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
axes[0].legend(fontsize=8)
fig.suptitle("Validation precision–recall · baseline configurations")
show()

With only eight validation events, a few ranks visibly change the curve. Assess sustained
precision over meaningful recall ranges alongside AP, rather than reading the zero-recall
endpoint as an actionable operating point.

### Confusion matrices and review burden

Compare all three saved decisions. Matrix shading is normalised within each actual class.
Each cell labels both the count and the within-class percentage. The queue chart uses
absolute counts to show the workload that those decisions would generate.

In [ ]:
plot_confusion(candidates)

rate_columns = ["Alert rate", "False discovery", "Precision", "Recall"]
display(format_table(operations, dict.fromkeys(rate_columns, "{:.2%}"), na_rep="—"))
plot_alerts(operations)

Unweighted logistic regression flagged **510 of 717 observations (71.1%)** and
**502 of those alerts were false (98.4%)** in the reference run. It caught all eight
observed events by sending most snapshots to review. The balanced fit sent 28 more
alerts and caught the same eight. This shows why observed recall alone gives an
incomplete picture of prioritisation.

## 11. Threshold analysis

The saved sweep records precision, recall and F1 at the original validation scores. It omits
alert counts. The chart therefore shows metric behaviour across thresholds, and the alert
table reports workload at the frozen thresholds using the saved confusion counts.
The threshold was selected by validation F1, so its measured F1 uses the same evidence
that selected the decision.

In [ ]:
sweep = pd.DataFrame(logistic["threshold_sweep"])
fig, ax = plt.subplots(figsize=(8, 4))
for field, colour in zip(("precision", "recall", "f1"), palette[1:4], strict=True):
    ax.plot(
        sweep["threshold"],
        sweep[field],
        label=field.upper() if field == "f1" else field.title(),
        color=colour,
    )
ax.axvline(logistic["threshold"], color=palette[0], ls="--", label="Selected validation threshold")
ax.set(
    title=f"Validation threshold trade-off · {logistic['name']}",
    xlabel="Score threshold",
    ylabel="Metric value",
)
ax.legend()
show()

## 12. Probability and calibration analysis

Calibration asks whether episodes with similar scores experience churn at similar rates.
Compare the dummy and unweighted logistic fit using fixed score bands. The score is the
bin's mean prediction; the rate is its observed fraction of positives. Counts and event
support matter because small bins can change sharply with one event. Brier score can be
dominated by the negative class when positives are rare.

In [ ]:
calibration_models = [candidates[0], logistic]
curve_colours = [palette[0], palette[1]]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.2, 1]})
calibration_tables = []
bin_axis = None
for item, colour, marker, offset in zip(
    calibration_models, curve_colours, ("o", "s"), (-0.18, 0.18), strict=True
):
    cal = pd.DataFrame(item["calibration"])
    occupied = cal[cal["count"] > 0]
    axes[0].scatter(
        occupied["mean_score"],
        occupied["observed_rate"],
        label=item["name"],
        color=colour,
        marker=marker,
        s=38,
    )
    bin_axis = cal["bin"]
    axes[1].bar(np.arange(len(cal)) + offset, cal["count"], width=0.35, color=colour)
    calibration_tables.append(cal.assign(Model=item["name"]))
axes[0].plot([0, 1], [0, 1], color="#6a7583", ls="--", lw=1, label="Equal score and event rate")
axes[0].set(
    title="Validation reliability in occupied bins",
    xlabel="Mean score",
    ylabel="Observed event rate",
    xlim=(0, 1),
    ylim=(0, 1),
)
axes[0].legend(fontsize=8)
axes[1].set(
    title="Validation observations per score bin",
    xlabel="Score band",
    ylabel="Episode observations",
    xticks=range(len(bin_axis)),
    xticklabels=bin_axis,
)
axes[1].tick_params(axis="x", labelrotation=24)
for axis in (axes[0].xaxis, axes[0].yaxis):
    axis.set_major_formatter(PercentFormatter(1))
show()
calibration_display = pd.concat(calibration_tables, ignore_index=True).rename(
    columns={
        "bin": "Score band",
        "count": "Rows",
        "positives": "Positive rows",
        "mean_score": "Mean score",
        "observed_rate": "Observed rate",
    }
)
display(
    format_table(
        calibration_display, {"Mean score": "{:.2%}", "Observed rate": "{:.2%}"}, na_rep="—"
    )
)

The dummy score stays near the observed prevalence and has the smallest Brier score.
The weighted fit has a much larger error on that scale. In the reference run, all
eight positives in the unweighted fit fell in the 0–1% bin. The support and the
score–rate mismatch leave little basis for using these scores as event probabilities.

## 13. Error analysis

Compare the logistic decision across observation quarter, episode age, billing frequency
and as-known usage coverage. Rate denominators are **all rows in each cohort**. The chart
compares alert and false-alert volume after accounting for cohort size. A cohort with few or
zero positives provides little evidence about recall.

In [ ]:
cohort_records = baseline["validation_cohorts"]

cohort = cohort_table(cohort_records)
display(format_table(cohort, {"Alert rate": "{:.1%}", "False alerts / all rows": "{:.1%}"}))
for _, dimension in cohort.groupby("Cohort"):
    assert dimension[
        ["Rows", "Positive rows", "True alerts", "False alerts", "Missed churn"]
    ].sum().tolist() == [chosen_metrics[name] for name in ("rows", "positives", "tp", "fp", "fn")]
fig, axes = plt.subplots(2, 2, figsize=(11.5, 6))
cohort_titles = {
    "quarter": "Observation quarter",
    "episode_age": "Episode age",
    "billing_frequency": "Billing frequency",
    "usage_coverage": "Usage coverage",
}
for ax, (key, label) in zip(axes.flat, cohort_titles.items(), strict=True):
    subset = cohort[cohort["Cohort"] == key]
    ax.barh(subset["Group"], subset["Alert rate"], color="#255889")
    ax.scatter(
        subset["False alerts / all rows"],
        subset["Group"],
        color="#a74932",
        marker="|",
        s=115,
        label="False alerts / all rows",
    )
    ax.set(title=label, xlabel="Share of cohort observations", xlim=(0, 1))
    ax.xaxis.set_major_formatter(PercentFormatter(1))
fig.suptitle("Validation alert rate by cohort · fitted decision held fixed")
axes[0, 0].legend(loc="lower right", fontsize=8)
show()

## 14. Baseline conclusions

In the reference run, the unweighted logistic model produced 114 false alerts in 157 annual
observations and 388 in 560 monthly observations. The groups had only two and six churn
observations respectively. This describes queue burden. It provides limited evidence
about differences between billing cohorts.

Logistic regression is a baseline for the nonlinear comparison. Its lower AP than prevalence
and large review queue limit the business case. Its calibration remains uncertain, with no
independent predictive evaluation.

In [ ]:
reference_ap = candidates[0]["validation"]["average_precision"]
lo, hi = baseline["validation_intervals"]["average_precision"]["interval_95"]
display(
    Markdown(
        f"The highest-ranked logistic configuration was **{logistic['name']}** (AP "
        f"{chosen_metrics['average_precision']:.4f}, prevalence reference {reference_ap:.4f}). "
        f"It generated **{chosen_metrics['fp']} false alerts** and missed "
        f"**{chosen_metrics['fn']}** of **{chosen_metrics['positives']}** "
        f"validation events at threshold **{logistic['threshold']:.5f}**. Its descriptive "
        f"95% account-bootstrap AP interval was **{lo:.4f}–{hi:.4f}**."
    )
)

The interval conditions on the fitted model and observed accounts. It omits model-selection uncertainty. The advanced comparison keeps the same population, target and evidence gate.